# Modeling shelter occupancy rates

This notebook trains three regression models -- Random Forest, XGBoost,
and Gradient Boosting -- to predict daily shelter occupancy rates.
We use a temporal train/test split to avoid data leakage, cross-validate
each model, compare performance, and tune the best candidate.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.model_selection import RandomizedSearchCV

sys.path.insert(0, '..')
from src.data_loader import load_and_prepare
from src.model import (
    encode_categorical, temporal_train_test_split, prepare_features_target,
    get_model, evaluate, get_feature_importance, DEFAULT_FEATURES,
    train_all_models
)

pd.set_option('display.max_columns', 50)

## 1. Load prepared data

In [ ]:
df = load_and_prepare(use_cache=True)
print(f"Loaded {len(df)} records")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Columns: {list(df.columns)}")

## 2. Encode categoricals and split

We encode `sheltertype` with a label encoder, then perform a temporal
train/test split. The most recent 20% of observations become the test set,
preserving time ordering.

In [ ]:
df_enc, encoders = encode_categorical(df, columns=['sheltertype'])
train_df, test_df = temporal_train_test_split(df_enc, test_fraction=0.2)

print(f"Train: {len(train_df)} rows, {train_df['date'].min().date()} to {train_df['date'].max().date()}")
print(f"Test:  {len(test_df)} rows, {test_df['date'].min().date()} to {test_df['date'].max().date()}")

In [ ]:
X_train, y_train = prepare_features_target(train_df, DEFAULT_FEATURES, 'occupancy_rate')
X_test, y_test = prepare_features_target(test_df, DEFAULT_FEATURES, 'occupancy_rate')

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"\nFeatures used: {list(X_train.columns)}")

## 3. Train Random Forest

In [ ]:
rf = get_model('random_forest')
rf.fit(X_train, y_train)

rf_train_metrics = evaluate(y_train, rf.predict(X_train))
rf_test_metrics = evaluate(y_test, rf.predict(X_test))

print("Random Forest")
print(f"  Train: {rf_train_metrics}")
print(f"  Test:  {rf_test_metrics}")

## 4. Train XGBoost

In [ ]:
xgb = get_model('xgboost')
xgb.fit(X_train, y_train)

xgb_train_metrics = evaluate(y_train, xgb.predict(X_train))
xgb_test_metrics = evaluate(y_test, xgb.predict(X_test))

print("XGBoost")
print(f"  Train: {xgb_train_metrics}")
print(f"  Test:  {xgb_test_metrics}")

## 5. Train Gradient Boosting Regressor

In [ ]:
gbr = get_model('gradient_boosting')
gbr.fit(X_train, y_train)

gbr_train_metrics = evaluate(y_train, gbr.predict(X_train))
gbr_test_metrics = evaluate(y_test, gbr.predict(X_test))

print("Gradient Boosting")
print(f"  Train: {gbr_train_metrics}")
print(f"  Test:  {gbr_test_metrics}")

## 6. Cross-validation with TimeSeriesSplit

Standard k-fold cross-validation is inappropriate for time series data
because it allows future information to leak into training folds.
`TimeSeriesSplit` ensures each validation fold is always strictly after
the training fold.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

models = {
    'RandomForest': get_model('random_forest'),
    'XGBoost': get_model('xgboost'),
    'GradientBoosting': get_model('gradient_boosting'),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=tscv,
                             scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name}: mean R2 = {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
# Visualise cross-validation results
cv_df = pd.DataFrame(cv_results)
cv_df['fold'] = range(1, len(cv_df) + 1)
cv_long = cv_df.melt(id_vars='fold', var_name='Model', value_name='R2')

fig = px.box(cv_long, x='Model', y='R2', points='all',
             title='Cross-validation R2 scores (TimeSeriesSplit, 5 folds)')
fig.update_layout(height=400)
fig.show()

## 7. Compare models on the held-out test set

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'Gradient Boosting'],
    'MAE': [rf_test_metrics['MAE'], xgb_test_metrics['MAE'], gbr_test_metrics['MAE']],
    'RMSE': [rf_test_metrics['RMSE'], xgb_test_metrics['RMSE'], gbr_test_metrics['RMSE']],
    'R2': [rf_test_metrics['R2'], xgb_test_metrics['R2'], gbr_test_metrics['R2']],
})

print(comparison.to_string(index=False))

best_model_name = comparison.loc[comparison['R2'].idxmax(), 'Model']
print(f"\nBest model by R2: {best_model_name}")

In [ ]:
fig = go.Figure()
for metric in ['MAE', 'RMSE']:
    fig.add_trace(go.Bar(name=metric, x=comparison['Model'], y=comparison[metric]))

fig.update_layout(barmode='group', title='Model comparison: MAE and RMSE (lower is better)',
                  yaxis_title='Error', height=400)
fig.show()

## 8. Hyperparameter tuning for the best model

We run a randomized search over key hyperparameters of the best-performing
model using `TimeSeriesSplit` for the internal cross-validation.

In [ ]:
# Tune XGBoost (adjust if a different model won above)
from xgboost import XGBRegressor

param_distributions = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
}

search = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions,
    n_iter=20,
    cv=TimeSeriesSplit(n_splits=3),
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)
print(f"\nBest params: {search.best_params_}")
print(f"Best CV R2:  {search.best_score_:.4f}")

In [ ]:
# Evaluate tuned model on the test set
tuned_pred = search.best_estimator_.predict(X_test)
tuned_metrics = evaluate(y_test, tuned_pred)
print(f"Tuned model test metrics: {tuned_metrics}")

print(f"\nImprovement over default XGBoost:")
print(f"  R2:   {xgb_test_metrics['R2']:.4f} -> {tuned_metrics['R2']:.4f}")
print(f"  MAE:  {xgb_test_metrics['MAE']:.4f} -> {tuned_metrics['MAE']:.4f}")

## 9. Save the best model

In [ ]:
from src.model import save_model

result = {
    'model': search.best_estimator_,
    'model_name': 'xgboost_tuned',
    'encoders': encoders,
    'feature_cols': list(X_train.columns),
    'target_col': 'occupancy_rate',
    'train_metrics': evaluate(y_train, search.best_estimator_.predict(X_train)),
    'test_metrics': tuned_metrics,
    'feature_importance': get_feature_importance(search.best_estimator_, list(X_train.columns)),
    'y_test': y_test.values,
    'y_pred_test': tuned_pred,
}

save_model(result, 'best_shelter_model.joblib')
print("Model saved successfully.")

## 10. Summary

Three models were trained and evaluated using a temporal split:

- **Random Forest** provides a solid baseline with minimal tuning.
- **XGBoost** and **Gradient Boosting** both achieve strong R2 scores.
- Cross-validation with `TimeSeriesSplit` confirms stable performance.
- Hyperparameter tuning via randomized search further improved the best model.

The next notebook performs a detailed evaluation of the selected model,
including residual analysis, feature importance, and error breakdowns
by shelter type.